In [1]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"


25/12/11 15:27:02 WARN Utils: Your hostname, Manojrams-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.219 instead (on interface en0)
25/12/11 15:27:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/manojrammopati/.ivy2/cache
The jars for the packages stored in: /Users/manojrammopati/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-azure added as a dependency
com.azure#azure-storage-blob added as a dependency
com.azure#azure-identity added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-af45e827-14f6-4a2e-bece-83b770a67efd;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central


:: loading settings :: url = jar:file:/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.fasterxml.jackson.core#jackson-core;2.12.7 in central
	found org.slf4j#slf4j-api;1.7.36 in central
	found com.microsoft.azure#azure-keyvault-core;1.0.0 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failureaccess;1.0 in central
	found com.google.guava#listenablefuture;9999.0-empty-to-avoid-conflict-with-guava in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.5.2 in central
	found com.google.errorprone#error_prone_annotations;2.2.0 in central
	found 

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [2]:

# Cell 0 — Optional: ensure Spark & paths helpers are available
# (same pattern you use in other notebooks)

import sys
import importlib
import os

# Adjust PROJECT_ROOT if needed
PROJECT_ROOT = "/Users/manojrammopati/Public/Projects/bupa_insurance_project"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Add Silver utils into path
silver_dir = os.path.join(PROJECT_ROOT, "_02_Silver")
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

import utils_silver
importlib.reload(utils_silver)

print("Loaded utils_silver from:", utils_silver.__file__)


✅ utils_silver.py loaded
✅ utils_silver.py loaded
Loaded utils_silver from: /Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver/utils_silver.py


# Cell 1 – Intro markdown (comment)
## ML Scoring – High Cost Claims (Is_High_Cost)

Goal: load the best high-cost model trained on ft_claims_risk_split and score
dq-valid claims, writing the scored output to Gold.


# Cell 2 – Imports & configuration

In [13]:
# Cell 2 — Imports & config

from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

STORAGE_ACCOUNT = "clientdatastorage"
GOLD_CONTAINER  = "golddata"
DB_GOLD         = "bupa_gold"

FT_CLAIMS_RISK_SPLIT_PATH = (
    f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/ft_claims_risk_split"
)

MODEL_BASE_PATH = (
    f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/models"
)

# 👇 Adjust to match the training notebook's "Saved best model to" line
MODEL_NAME = "claims_high_cost_best"

MODEL_PATH = f"{MODEL_BASE_PATH}/{MODEL_NAME}"

SCORED_BASE_PATH = (
    f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/scored_claims"
)
SCORED_PATH = f"{SCORED_BASE_PATH}/high_cost_claims"

ML_MON_PATH = f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/ml_monitoring_view"

print("Feature table      :", FT_CLAIMS_RISK_SPLIT_PATH)
print("High-cost model    :", MODEL_PATH)
print("Scored output path :", SCORED_PATH)
print("ML monitoring view path :", ML_MON_PATH)


Feature table      : abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_claims_risk_split
High-cost model    : abfss://golddata@clientdatastorage.dfs.core.windows.net/models/claims_high_cost_best
Scored output path : abfss://golddata@clientdatastorage.dfs.core.windows.net/scored_claims/high_cost_claims
ML monitoring view path : abfss://golddata@clientdatastorage.dfs.core.windows.net/ml_monitoring_view


# Cell 3 – Load feature table & pick rows to score

In [5]:
# Cell 3 — Load ft_claims_risk_split and select scoring population

LABEL_COL = "Is_High_Cost"

claims_all = (
    spark.read
         .format("delta")
         .load(FT_CLAIMS_RISK_SPLIT_PATH)
)

print("Total rows in ft_claims_risk_split:", claims_all.count())
claims_all.printSchema()

claims_scoring = (
    claims_all
        .filter((F.col("dq_money_valid") == 1) & (F.col("dq_date_valid") == 1))
)

print("Rows to score (dq-valid):", claims_scoring.count())
claims_scoring.select("Claim_ID", "dataset_split", LABEL_COL).show(5, truncate=False)


25/12/11 15:32:38 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Total rows in ft_claims_risk_split: 558211
root
 |-- Claim_ID: string (nullable = true)
 |-- Member_Key: string (nullable = true)
 |-- Provider_ID: string (nullable = true)
 |-- Claim_Type_Name: string (nullable = true)
 |-- Claim_Type_Code: string (nullable = true)
 |-- Claim_Status: string (nullable = true)
 |-- Claim_Amount_GBP: double (nullable = true)
 |-- Payout_GBP: double (nullable = true)
 |-- Payout_to_Amount_Ratio: double (nullable = true)
 |-- Days_To_Settle: integer (nullable = true)
 |-- High_Cost_Claim_Flag: integer (nullable = true)
 |-- Claim_Fraud_Label: integer (nullable = true)
 |-- Provider_Fraud_Label: integer (nullable = true)
 |-- Provider_Risk_Tier: string (nullable = true)
 |-- dq_money_valid: integer (nullable = true)
 |-- dq_date_valid: integer (nullable = true)
 |-- Is_Fraudulent_Claim: integer (nullable = true)
 |-- Is_High_Cost: integer (nullable = true)
 |-- dataset_split: string (nullable = true)



Rows to score (dq-valid): 558211


+---------+-------------+------------+
|Claim_ID |dataset_split|Is_High_Cost|
+---------+-------------+------------+
|CLM110013|train        |0           |
|CLM110020|train        |1           |
|CLM110025|train        |0           |
|CLM110026|train        |1           |
|CLM110041|train        |0           |
+---------+-------------+------------+
only showing top 5 rows



# Cell 4 – Features & null-handling (same as fraud / training)

In [6]:
# Cell 4 — Define feature columns & null-handling

cols = claims_scoring.columns

numeric_candidates = [
    "Claim_Amount_GBP",
    "Payout_GBP",
    "Payout_to_Amount_Ratio",
    "Days_To_Settle",
]

categorical_candidates = [
    "Claim_Type_Name",
    "Claim_Status",
    "Claim_Type_Code",
    "Provider_Risk_Tier",
    "Provider_ID",
]

numeric_cols = [c for c in numeric_candidates if c in cols]
cat_cols     = [c for c in categorical_candidates if c in cols]

print("Numeric features   :", numeric_cols)
print("Categorical features:", cat_cols)

id_cols = [c for c in ["Claim_ID", "Member_Key", "Provider_ID"] if c in cols]

def prep_nulls(df: DataFrame) -> DataFrame:
    d = df
    for c in numeric_cols:
        d = d.withColumn(c, F.when(F.col(c).isNull(), F.lit(0.0)).otherwise(F.col(c)))
    for c in cat_cols:
        d = d.withColumn(c, F.when(F.col(c).isNull(), F.lit("Unknown")).otherwise(F.col(c)))
    return d


Numeric features   : ['Claim_Amount_GBP', 'Payout_GBP', 'Payout_to_Amount_Ratio', 'Days_To_Settle']
Categorical features: ['Claim_Type_Name', 'Claim_Status', 'Claim_Type_Code', 'Provider_Risk_Tier', 'Provider_ID']


# Cell 5 – Load best high-cost model & score

In [7]:
# Cell 5 — Load high-cost model & score

print("Loading high-cost model from:", MODEL_PATH)
loaded_model = PipelineModel.load(MODEL_PATH)

features_prepped = prep_nulls(claims_scoring)

scored_raw = loaded_model.transform(features_prepped)

scored = (
    scored_raw
        .withColumn("prob_arr", vector_to_array("probability"))
        .withColumn("high_cost_prob", F.col("prob_arr")[1])
)

keep_cols = (
    id_cols
    + [LABEL_COL]
    + [
        "high_cost_prob",
        "prediction",
        "Claim_Type_Name",
        "Claim_Status",
        "Claim_Amount_GBP",
        "Payout_GBP",
        "Payout_to_Amount_Ratio",
        "Provider_Risk_Tier",
        "dataset_split",
    ]
)

scored_final = scored.select(*[c for c in keep_cols if c in scored.columns])

print("Scored schema:")
scored_final.printSchema()
scored_final.show(10, truncate=False)


Loading high-cost model from: abfss://golddata@clientdatastorage.dfs.core.windows.net/models/claims_high_cost_best


Scored schema:
root
 |-- Claim_ID: string (nullable = true)
 |-- Member_Key: string (nullable = true)
 |-- Provider_ID: string (nullable = true)
 |-- Is_High_Cost: integer (nullable = true)
 |-- high_cost_prob: double (nullable = true)
 |-- prediction: double (nullable = false)
 |-- Claim_Type_Name: string (nullable = true)
 |-- Claim_Status: string (nullable = true)
 |-- Claim_Amount_GBP: double (nullable = true)
 |-- Payout_GBP: double (nullable = true)
 |-- Payout_to_Amount_Ratio: double (nullable = true)
 |-- Provider_Risk_Tier: string (nullable = true)
 |-- dataset_split: string (nullable = true)



+---------+----------+-----------+------------+--------------------+----------+---------------+------------+------------------+----------+----------------------+------------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Is_High_Cost|high_cost_prob      |prediction|Claim_Type_Name|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|Provider_Risk_Tier|dataset_split|
+---------+----------+-----------+------------+--------------------+----------+---------------+------------+------------------+----------+----------------------+------------------+-------------+
|CLM110013|BENE15435 |PRV51790   |0           |0.021246719556712867|0.0       |Travel         |Settled     |244.07511199775644|200.0     |0.8194198841618822    |Low risk          |train        |
|CLM110020|BENE53030 |PRV53679   |1           |0.31883079704452943 |0.0       |Hospital       |Settled     |4117.6845027538575|2600.0    |0.6314228295686937    |Low risk          |train        |
|CLM110025|BENE69397 |PRV

# Cell 6 – Write scored high-cost results to Gold + register table

In [9]:
DB_GOLD = "bupa_gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD}")

DataFrame[]

In [15]:
# Cell 6 — Persist scored high-cost claims to Gold & create table

(
    scored_final
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")  # or "append" for history
        .save(SCORED_PATH)
)

(
    scored_final
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")  # or "append" for history
        .save(ML_MON_PATH)
)

print("✅ Wrote scored high-cost claims to:", SCORED_PATH, "and to ML monitoring view path:", ML_MON_PATH)

TABLE_NAME = "scored_claims_high_cost"

spark.sql(f"DROP TABLE IF EXISTS {DB_GOLD}.{TABLE_NAME}")

spark.sql(f"""
    CREATE TABLE {DB_GOLD}.{TABLE_NAME}
    USING DELTA
    LOCATION '{SCORED_PATH}'
""")

print(f"✅ Registered table: {DB_GOLD}.{TABLE_NAME}")
spark.table(f"{DB_GOLD}.{TABLE_NAME}").show(10, truncate=False)


✅ Wrote scored high-cost claims to: abfss://golddata@clientdatastorage.dfs.core.windows.net/scored_claims/high_cost_claims and to ML monitoring view path: abfss://golddata@clientdatastorage.dfs.core.windows.net/ml_monitoring_view
✅ Registered table: bupa_gold.scored_claims_high_cost


+---------+----------+-----------+------------+--------------------+----------+---------------+------------+------------------+----------+----------------------+------------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Is_High_Cost|high_cost_prob      |prediction|Claim_Type_Name|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|Provider_Risk_Tier|dataset_split|
+---------+----------+-----------+------------+--------------------+----------+---------------+------------+------------------+----------+----------------------+------------------+-------------+
|CLM110013|BENE15435 |PRV51790   |0           |0.021246719556712867|0.0       |Travel         |Settled     |244.07511199775644|200.0     |0.8194198841618822    |Low risk          |train        |
|CLM110020|BENE53030 |PRV53679   |1           |0.31883079704452943 |0.0       |Hospital       |Settled     |4117.6845027538575|2600.0    |0.6314228295686937    |Low risk          |train        |
|CLM110025|BENE69397 |PRV